In [0]:
%python
from pyspark.sql.functions import expr,col,current_timestamp,current_date ,lit,sum , max , coalesce,row_number
from pyspark.sql.window import Window
from datetime import datetime

class OrderProcessor:
    def __init__(self):

        self.spark = spark
        self.checkpointLocationquantity = '/Volumes/dev/spark_db/data/checkpoint/Quantity/'
        self.checkpointlocation_txn = '/Volumes/dev/spark_db/data/checkpoint/Transaction/'

        self.process_dt = datetime.now()

    def read_staging_df(self):

        return (
            self.spark.read.table("dev.spark_db.order_stg")
        )

    def transaction(self,stg_df):

        account = self.spark.read.table("dev.spark_db.account")
        asset = self.spark.read.table("dev.spark_db.asset")

        max_transaction_id = (
                                self.spark.read.table("dev.spark_db.transaction")
                                                .select(coalesce(max(col("transaction_id")),lit(10000))
                                                            .alias("max_seq")).collect()[0]['max_seq']
                            )

        txn_df = (
            stg_df.alias("sd").join(account.alias("ac"), expr("sd.account_number = ac.account_number"), "inner")
                              .join(asset.alias("as") , expr("sd.asset_external_id = as.asset_external_id"),"inner")
                              .select("ac.account_number",
                                      "as.asset_external_id",
                                      "sd.order_type",
                                      "as.asset_currency",
                                      "sd.asset_rate",
                                      "sd.quantity",
                                      "sd.asset_rate",
                                      "sd.order_amt",
                                      col("sd.as_of_dt").alias("transaction_dt"),
                                      expr(" 1 as transaction_ver"),
                                      current_timestamp().alias("transaction_dml")
                                      )
                              .withColumn(
                                  "transaction_id" , 
                                  row_number().over(Window.orderBy(lit(1))) + max_transaction_id          
                                           )
        )

        return txn_df
    
    def write_to_transaction(self,stg_df):

        return (
            stg_df.writeStream.format("delta")
                              .queryName("Transaction_Query")
                              .outputMode("append")
                              .option("checkpointLocation" ,self.checkpointlocation_txn)
                              .trigger( availableNow = True)
                              .toTable("dev.spark_db.transaction")
        )

    def get_asset_rate(self ,asset_list):

        asset_rate = self.spark.read.table("asset_rate").alias("ar")

        get_rate = ( asset_list.alias("al").join(asset_rate,
                                               on = expr("""
                                                         ar.asset_external_id = al.asset_external_id
                                                         and 
                                                         ar.as_of_dt = al.as_of_dt
                                                            """),
                                               how = 'inner')
                                        .select("ar.asset_external_id",
                                                "ar.as_of_dt",
                                                "ar.asset_rate")
        )

        return get_rate


    def quantity_process(self,txn_df):

        from delta.tables import DeltaTable
        from pyspark.sql.functions import broadcast,expr

        quantity = DeltaTable.forPath(self.spark , "/Volumes/dev/spark_db/data/table/Quantity/")

        quantity_to_df = quantity.toDF()

        account_list = txn_df.select("account").distinct()

        txn_df = ( 
                  txn_df.groupBy("account_number", "asset_external_id")
                        .agg( sum("quantity").alias("total_quantity"))
        )

        asset_list = txn_df.select("asset_external_id",current_date().alias("as_of_dt")).distinct()

        asset_rate = self.get_asset_rate(asset_list)

        ( quantity.alias("qty").merge(txn_df.alias("stg"),expr('''
                                                                qty.account_number=stg.account_number
                                                                and 
                                                                qty.asset_external_id=stg.asset_external_id
                                                                '''))
                                .whenMatchedUpdate(
                                  set =   {
                                        "available_quantity" : expr("stg.total_quantity + qty.quantity")
                                     }
                                ).whenNotMatchedInsert(
                                 values =    {
                                        "account_number" : "stg.account_number",
                                        "asset_external_id" : "stg.asset_external_id",
                                        "quantity" : "stg.quantity",
                                        "quantity_dml" : current_timestamp()
                                    }
                                ).execute()
        )


    def write_to_quantity_dynamodb(self, batch_df , batch_id):

        def update_partition(rows):
            import boto3 

            dynamodb = boto3.resource("dynamodb")

            quantity = dynamodb.Table("quantity")

            for row in rows:
                quantity.update_item(
                    key = {
                        "account_number" : row["account_number"],
                        "asset_external_id" : row["asset_external_id"]
                    },
                    UpdateExpression = '''
                    set #qu = :quantity,
                        #ab = :available_balance
                    ''',
                    ExpressionAttributeNames = {
                        "#qu" : "quantity",
                        "#av" : "available_balance"
                    },
                    ExpressionAttributeValues = {
                        ":quantity" : row['quantity'],
                        ":available_balance" : row['available_balance']
                    }
                    # default upsert
                    #if only insert
                        #ConditionExpression = 'attribute_not_exists(account_number,asset_external_id)'
                    #if only update
                        #ConditionExpression = 'attribute_exists(account_number,asset_external_id)'

                )

        batch_df.foreachPartition(update_partition)

    def write_to_quantity(self , order_stg):

        quantity_updated = self.spark.read.table("quantity")

        quantity_df = ( quantity_updated.alias("qu").join(order_stg.alias("os"),
                                                        expr('''
                                                            qu.account_number=os.account_number
                                                            and 
                                                            qu.asset_external_id = os.external_id
                                                                                   '''),
                                                        "inner"
                                                        )
                                    .select("qu.account_number",
                                            "qu.asset_external_id",
                                            "qu.quantity",
                                            current_timestamp().alias("quantity_dml"))
        )
                                    
        return (quantity_df.writeStream.format("delta")
                                .queryName("quantity_query")
                                .foreachBatch(self.write_to_quantity_dynamodb)
                                .option("checkpointLocation" , self.checkpointLocationquantity)
                                .trigger(availableNow = True)
                                .start()
        )

    def main(self):

        read_stg = self.read_staging_df()

        read_stg.display()

        #transaction_query = self.write_to_transaction(read_stg)

        display(read_stg,checkpointLocation = '/Volumes/dev/spark_db/data/checkpoint/Display/')

        quantity_val = self.quantity_process(self.transaction(read_stg))


if __name__ == '__main__':
    op = OrderProcessor()
    op.main()
                            




In [0]:
select * from dev.spark_db.quantity;

In [0]:
%python

from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DecimalType

data = [
        ('97940-69402',	'HDFCBANK'),
        ('06932-99420',	'TCS'),
        ('126934-24574',	'HDFCBANK'),
        ('06932-99420',	'RELIANCE'),
        ('07942-19393',	'RELIANCE'),
        ('907958-17375',	'AMZN'),
        ('97940-69402',	'AAPL'),
        ('468902-67386',	'RELIANCE'),
       ('468902-67386',	'NVDA'),
        ('12059-49504',	'GOOGL'),
        ('896995-28486',	'GOOGL')
    ]
    
schema = StructType(
                [
                StructField("account_number" , StringType() , False),
                StructField("asset_external_id" , StringType() , False)
                ]
    )

quantity_df = spark.createDataFrame(data = data , schema = schema)

quantity_df = quantity_df.withColumns(
    {
        "available_quantity" : lit(100.00).cast(DecimalType(18,2)),
        "quantity_dml" : current_timestamp()
    }
    )

quantity_df.write \
    .format("delta") \
    .mode("append") \
    .option("path", "/Volumes/dev/spark_db/data/checkpoint/Quantity/") \
    .save()

In [0]:
%python

from pyspark.sql.functions import max,expr,col,lit,coalesce,row_number
from pyspark.sql.window import Window

data = [('raju' , 100) , ('ramu' , 47)]
schema = "name string , age int"

df = spark.createDataFrame(data , schema)

def sequence(seq):
    return seq.collect()[0]['no']

seq = spark.read.table("dev.spark_db.employee").select(coalesce(max(col("employee_id")), lit(1000)).alias("no"))

seq = df.withColumn("employee_id" , row_number().over(Window.orderBy(lit(1)))+ sequence(seq))

seq.display()

In [0]:
create or replace table dev.spark_db.employee(
    employee_id int,
    e_name string,
    age int
)
using delta;

In [0]:
create or replace table dev.spark_db.transaction(
    transaction_id bigint,
    account_number string,
    asset_exteranl_id string,
    order_typr string,
    asset_currency string,
    asset_rate decimal(18,2),
    asset_rate decimal(18,2),
    order_amt decimal(18,2),
    transaction_dt date,
    transaction_ver smallint,
    transaction_dml timestamp

) using delta;

In [0]:
select * from dev.spark_db.quantity;